# Binning and binarization
Rebuilt for scikit-learn 1.9 and pandas 3. Data: `data/titanic_train.csv` (the Kaggle Titanic training file, 891 rows).

In [1]:
# libraries for data, splitting, the model, scoring and the two transformers
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import KBinsDiscretizer, Binarizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

## 1. Discretization with KBinsDiscretizer

In [2]:
# keep three columns and drop the 177 rows with a missing age
df = pd.read_csv("data/titanic_train.csv", usecols=["Age", "Fare", "Survived"])
df = df.dropna()
print(df.shape)
df.head()

(714, 3)


,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833
2,1,26.0,7.9250
3,1,35.0,53.1000
4,0,35.0,8.0500


In [3]:
# inputs and target, then an 80/20 split
X = df[["Age", "Fare"]]
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
# baseline: a decision tree on the raw numbers
# random_state=0 fixes how the tree breaks ties, so the numbers repeat
clf = DecisionTreeClassifier(random_state=0)
clf.fit(X_train, y_train)
print("test accuracy:", accuracy_score(y_test, clf.predict(X_test)))
print("10-fold CV:", cross_val_score(DecisionTreeClassifier(random_state=0), X, y, cv=10).mean())

test accuracy: 0.6223776223776224


10-fold CV: 0.6302621283255085


In [5]:
# one discretizer per column: 15 equal-frequency bins, output = bin number
kbin_age = KBinsDiscretizer(n_bins=15, encode="ordinal", strategy="quantile")
kbin_fare = KBinsDiscretizer(n_bins=15, encode="ordinal", strategy="quantile")
trf = ColumnTransformer([("age", kbin_age, ["Age"]), ("fare", kbin_fare, ["Fare"])])

# learn the edges on the training set only, then apply them to both sets
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

In [6]:
# the learned edges: 16 edges make 15 bins
print(trf.named_transformers_["age"].bin_edges_[0])
print(trf.named_transformers_["fare"].bin_edges_[0])

[ 0.42  6.   16.   19.   21.   23.   25.   28.   30.   32.   35.   38.
 42.   47.   54.   80.  ]
[  0.       7.25     7.775    7.8958   8.1583  10.5     13.      14.4542
  18.75    26.      26.55    31.275   51.4792  76.2917 108.9    512.3292]


In [7]:
# each value, its bin number, and the range of that bin
# KBinsDiscretizer's bins include their left edge, so right=False makes pd.cut agree
age_edges = trf.named_transformers_["age"].bin_edges_[0]
fare_edges = trf.named_transformers_["fare"].bin_edges_[0]
output = pd.DataFrame({
    "age": X_train["Age"], "age_trf": X_train_trf[:, 0],
    "age_range": pd.cut(X_train["Age"], age_edges, right=False),
    "fare": X_train["Fare"], "fare_trf": X_train_trf[:, 1],
    "fare_range": pd.cut(X_train["Fare"], fare_edges, right=False),
})
output.sample(5, random_state=1)

,age,age_trf,age_range,fare,fare_trf,fare_range
3,35.0,10.0,"[35.0, 38.0)",53.1000,12.0,"[51.479, 76.292)"
192,19.0,3.0,"[19.0, 21.0)",7.8542,2.0,"[7.775, 7.896)"
84,17.0,2.0,"[16.0, 19.0)",10.5000,5.0,"[10.5, 13.0)"
135,23.0,5.0,"[23.0, 25.0)",15.0458,7.0,"[14.454, 18.75)"
537,30.0,8.0,"[30.0, 32.0)",106.4250,13.0,"[76.292, 108.9)"


In [8]:
# the same tree on the bin numbers
clf = DecisionTreeClassifier(random_state=0)
clf.fit(X_train_trf, y_train)
print("test accuracy:", accuracy_score(y_test, clf.predict(X_test_trf)))

# cross-validation: binning must go inside a pipeline,
# so each fold learns its edges from its own training part
pipe = make_pipeline(trf, DecisionTreeClassifier(random_state=0))
print("10-fold CV:", cross_val_score(pipe, X, y, cv=10).mean())

test accuracy: 0.6363636363636364


10-fold CV: 0.675293427230047


## 2. Trying every strategy and number of bins

In [9]:
# one function: bin both columns, report the CV accuracy, plot before and after
def discretize(bins, strategy):
    trf = ColumnTransformer([
        ("age", KBinsDiscretizer(n_bins=bins, encode="ordinal", strategy=strategy), ["Age"]),
        ("fare", KBinsDiscretizer(n_bins=bins, encode="ordinal", strategy=strategy), ["Fare"]),
    ])
    pipe = make_pipeline(trf, DecisionTreeClassifier(random_state=0))
    print(strategy, bins, "CV accuracy:", round(cross_val_score(pipe, X, y, cv=10).mean(), 4))

    X_trf = trf.fit_transform(X)
    fig = make_subplots(2, 2, subplot_titles=["Age before", "Age after", "Fare before", "Fare after"])
    fig.add_histogram(x=X["Age"], row=1, col=1)
    fig.add_histogram(x=X_trf[:, 0], marker_color="#E45756", row=1, col=2)
    fig.add_histogram(x=X["Fare"], row=2, col=1)
    fig.add_histogram(x=X_trf[:, 1], marker_color="#E45756", row=2, col=2)
    fig.update_layout(showlegend=False, height=550, title=f"{strategy}, {bins} bins")
    fig.show()

discretize(10, "quantile")

quantile 10 CV accuracy: 0.6753


In [10]:
discretize(10, "uniform")

uniform 10 CV accuracy: 0.6865


In [11]:
discretize(5, "kmeans")

kmeans 5 CV accuracy: 0.6781


In [12]:
# all three strategies side by side, 5 bins on the training Fare column: edges and rows per bin
for s in ["uniform", "quantile", "kmeans"]:
    kb = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy=s).fit(X_train[["Fare"]])
    counts = np.bincount(kb.transform(X_train[["Fare"]])[:, 0].astype(int), minlength=5)
    print(f"{s:9s} edges {np.round(kb.bin_edges_[0], 1)}  rows per bin {counts}")

uniform   edges [  0.  102.5 204.9 307.4 409.9 512.3]  rows per bin [530  26  14   0   1]
quantile  edges [  0.    7.9  13.   26.   51.5 512.3]  rows per bin [ 95 126 116 119 115]


kmeans    edges [  0.   42.4 100.6 186.5 376.6 512.3]  rows per bin [448  82  26  14   1]


In [13]:
# a table of CV accuracies for every strategy and number of bins
rows = []
for s in ["uniform", "quantile", "kmeans"]:
    for n in [5, 10, 15]:
        trf = ColumnTransformer([
            ("age", KBinsDiscretizer(n_bins=n, encode="ordinal", strategy=s), ["Age"]),
            ("fare", KBinsDiscretizer(n_bins=n, encode="ordinal", strategy=s), ["Fare"]),
        ])
        pipe = make_pipeline(trf, DecisionTreeClassifier(random_state=0))
        rows.append((s, n, cross_val_score(pipe, X, y, cv=10).mean()))
pd.DataFrame(rows, columns=["strategy", "bins", "cv_accuracy"]).pivot(index="bins", columns="strategy").round(3)

cv_accuracy                 
strategy      kmeans quantile uniform
bins                                 
5              0.678    0.677   0.632
10             0.665    0.675   0.686
15             0.663    0.675   0.653

## 3. Custom binning with domain knowledge

In [14]:
# our own edges: child below 18, adult 18 to 59, senior 60 and over
age_group = pd.cut(df["Age"], bins=[0, 18, 60, np.inf], right=False,
                   labels=["child", "adult", "senior"])
pd.concat([df["Age"], age_group.rename("age_group")], axis=1).head()

,Age,age_group
0,22.0,adult
1,38.0,adult
2,26.0,adult
3,35.0,adult
4,35.0,adult


In [15]:
# survival rate per group
df.groupby(age_group, observed=True)["Survived"].agg(["mean", "size"])

,mean,size
Age,,
child,0.539823,113
adult,0.386087,575
senior,0.269231,26


## 4. Binarization

In [16]:
# family = siblings/spouses + parents/children on board
df2 = pd.read_csv("data/titanic_train.csv", usecols=["Age", "Fare", "SibSp", "Parch", "Survived"]).dropna()
df2["family"] = df2["SibSp"] + df2["Parch"]
df2 = df2.drop(columns=["SibSp", "Parch"])
X2 = df2.drop(columns=["Survived"])
y2 = df2["Survived"]
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)
X2_train.head()

,Age,Fare,family
328,31.0,20.5250,2
73,26.0,14.4542,1
253,30.0,16.1000,1
719,33.0,7.7750,0
666,25.0,13.0000,0


In [17]:
# baseline without binarization
clf = DecisionTreeClassifier(random_state=0)
clf.fit(X2_train, y2_train)
print("test accuracy:", accuracy_score(y2_test, clf.predict(X2_test)))
print("10-fold CV:", cross_val_score(DecisionTreeClassifier(random_state=0), X2, y2, cv=10).mean())

test accuracy: 0.6083916083916084
10-fold CV: 0.6541471048513302


In [18]:
# family > 0 becomes 1 (travelling with family), family = 0 stays 0 (alone)
trf = ColumnTransformer([("bin", Binarizer(threshold=0), ["family"])], remainder="passthrough")
trf.set_output(transform="pandas")
X2_train_trf = trf.fit_transform(X2_train)
X2_test_trf = trf.transform(X2_test)
X2_train_trf.head()

,bin__family,remainder__Age,remainder__Fare
328,1,31.0,20.5250
73,1,26.0,14.4542
253,1,30.0,16.1000
719,0,33.0,7.7750
666,0,25.0,13.0000


In [19]:
# the same tree on the binarized data
clf = DecisionTreeClassifier(random_state=0)
clf.fit(X2_train_trf, y2_train)
print("test accuracy:", accuracy_score(y2_test, clf.predict(X2_test_trf)))
pipe = make_pipeline(trf, DecisionTreeClassifier(random_state=0))
print("10-fold CV:", cross_val_score(pipe, X2, y2, cv=10).mean())

test accuracy: 0.6153846153846154


10-fold CV: 0.6261737089201878


In [20]:
# survival rate: alone (0) against with family (1)
df2.groupby(df2["family"] > 0)["Survived"].agg(["mean", "size"])

,mean,size
family,,
False,0.321782,404
True,0.516129,310


### Binarization with a linear model
A linear model draws one straight trend through `family`, but survival rises then falls with family size. The yes/no split (alone or not) gives it the step it cannot draw. Logistic regression on `family` alone, raw against binarized; 10-fold CV averaged over 10 shuffles.

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

def cv_mean(pipe, X, y, scoring):
    # 10-fold CV, averaged over 10 different shuffles
    return np.mean([cross_val_score(pipe, X, y, scoring=scoring,
                    cv=StratifiedKFold(10, shuffle=True, random_state=k)).mean()
                    for k in range(10)])

Xf = X2[["family"]]
raw_lr = make_pipeline(LogisticRegression())
bin_lr = make_pipeline(ColumnTransformer([("bin", Binarizer(threshold=0), ["family"])]),
                       LogisticRegression())
for name, pipe in [("family as a number", raw_lr), ("family binarized", bin_lr)]:
    print(name, "accuracy:", round(cv_mean(pipe, Xf, y2, "accuracy"), 3),
          "ROC AUC:", round(cv_mean(pipe, Xf, y2, "roc_auc"), 3))

family as a number accuracy: 0.584 ROC AUC: 0.585


family binarized accuracy: 0.601 ROC AUC: 0.599


In [22]:
# a value equal to the threshold becomes 0: only values strictly above it become 1
Binarizer(threshold=127.5).fit_transform([[0, 100, 127.5, 128, 255]])

array([[0., 0., 0., 1., 1.]])